[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shashvindu/gonu-jha-ai-master-jee/blob/main/SQL/sql_01_second_highest_salary.ipynb) - run this notebook in your browser, nothing to install.

# SQL #1: Find the Second Highest Salary

**Gonu Jha AI Master Jee** · Interview prep notebook for the video *Find the Second Highest Salary* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. The question and the tricky cases interviewers test (duplicates, NULL)
2. Solution 1: subquery + `MAX` (works on every database)
3. Solution 2: window function `DENSE_RANK`
4. **Why `LIMIT 1 OFFSET 1` gives the wrong answer**
5. `ROW_NUMBER` vs `RANK` vs `DENSE_RANK` side by side
6. The general **Nth highest salary** query and the per-department version

In [1]:
# Everything runs on SQLite, which is built into Python - nothing to install.
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")

def sql(query):
    """Run a query and show the result as a table (a SQL NULL is shown as <NA>)."""
    return pd.read_sql_query(query, con).convert_dtypes()

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.50.4


## 1. Key definitions

| Term | Meaning |
|---|---|
| **Aggregate function** | Turns many rows into one value: `MAX`, `MIN`, `SUM`, `COUNT`, `AVG` |
| **Subquery** | A query inside another query, e.g. inside `WHERE salary < (SELECT ...)` |
| **DISTINCT** | Removes duplicate values |
| **NULL** | "No value". `MAX` over zero rows returns `NULL` |
| **Window function** | Computes a value for each row using other rows, without collapsing them: `RANK() OVER (...)` |
| **`DENSE_RANK`** | Ranks with no gaps after ties: 80000 → 1, 80000 → 1, 60000 → **2** |

> **Gonu Jha's story:** the prize was for the second-*best* wrestler, not the second man in line. Two wrestlers tied for first - so "second" must skip duplicates.

## 2. The question
Table `Employee(id, name, salary)`. Return the **second highest DISTINCT** salary, or `NULL` if there is none.

In [2]:
con.executescript("""
DROP TABLE IF EXISTS Employee;
CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER);
INSERT INTO Employee VALUES (1, 'Amit', 50000), (2, 'Priya', 80000), (3, 'Rahul', 80000), (4, 'Sneha', 60000);
""")
sql("SELECT * FROM Employee")

,id,name,salary
0,1,Amit,50000
1,2,Priya,80000
2,3,Rahul,80000
3,4,Sneha,60000


Priya and Rahul are tied at 80,000. The highest is 80,000, so the correct answer is **60,000**.

## 3. Think before you write
1. Find the highest salary with `MAX`
2. Keep only salaries **below** that highest
3. Take `MAX` again on what is left
4. Bonus: `MAX` on zero rows gives `NULL` automatically - the NULL case is handled for free

## 4. Solution 1: subquery + MAX

In [3]:
sql("""
SELECT MAX(salary) AS second_highest_salary
FROM Employee
WHERE salary < (
    SELECT MAX(salary)
    FROM Employee
)
""")

,second_highest_salary
0,60000


Step by step - first the inner query, then the filtered rows:

In [4]:
display(sql("SELECT MAX(salary) AS highest FROM Employee"))
sql("SELECT * FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

,highest
0,80000


,id,name,salary
0,1,Amit,50000
1,4,Sneha,60000


## 5. Solution 2: DENSE_RANK

In [5]:
sql("""
SELECT MAX(salary) AS second_highest_salary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
) ranked
WHERE rnk = 2
""")

,second_highest_salary
0,60000


Why `MAX(salary)` around it? If several people share rank 2, it still returns one row - and if nobody has rank 2 it returns `NULL` instead of an empty result.

## 6. ROW_NUMBER vs RANK vs DENSE_RANK

In [6]:
sql("""
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_number,
       RANK()       OVER (ORDER BY salary DESC) AS rank,
       DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rank
FROM Employee
""")

,name,salary,row_number,rank,dense_rank
0,Priya,80000,1,1,1
1,Rahul,80000,2,1,1
2,Sneha,60000,3,3,2
3,Amit,50000,4,4,3


| Function | Ties | 60,000 gets | Use for "2nd highest salary"? |
|---|---|---|---|
| `ROW_NUMBER` | Breaks ties arbitrarily | 3 | No |
| `RANK` | Same rank, then **skips** (1, 1, 3) | 3 | No - there is no rank 2! |
| `DENSE_RANK` | Same rank, **no gaps** (1, 1, 2) | 2 | **Yes** |

## 7. Why `LIMIT 1 OFFSET 1` fails

In [7]:
sql("SELECT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

,salary
0,80000


It returns **80,000** - the second *row*, not the second *distinct salary*. Adding `DISTINCT` fixes the duplicates, but on a one-row table it returns **no row at all** instead of `NULL`:

In [8]:
display(sql("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1"))

con.executescript("""
DROP TABLE IF EXISTS OneEmployee;
CREATE TABLE OneEmployee (id INTEGER, name TEXT, salary INTEGER);
INSERT INTO OneEmployee VALUES (1, 'Amit', 50000);
""")
print("LIMIT/OFFSET on a one-row table:")
display(sql("SELECT DISTINCT salary FROM OneEmployee ORDER BY salary DESC LIMIT 1 OFFSET 1"))
print("Subquery + MAX on a one-row table:")
sql("SELECT MAX(salary) AS second_highest_salary FROM OneEmployee WHERE salary < (SELECT MAX(salary) FROM OneEmployee)")

,salary
0,60000


LIMIT/OFFSET on a one-row table:


,salary


Subquery + MAX on a one-row table:


,second_highest_salary
0,None


## 8. Generalize: the Nth highest salary

In [9]:
def nth_highest(n):
    return sql(f"""
    SELECT MAX(salary) AS salary_rank_{n}
    FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM Employee)
    WHERE rnk = {n}
    """)

for n in (1, 2, 3, 4):
    display(nth_highest(n))

,salary_rank_1
0,80000


,salary_rank_2
0,60000


,salary_rank_3
0,50000


,salary_rank_4
0,None


## 9. Interview talking points
- Say the **duplicate case** out loud before the interviewer asks ("two people earn 80,000, so I need DISTINCT values").
- Say the **NULL case**: what if the table has only one salary?
- Offer both solutions: subquery + MAX (simple, works everywhere) and DENSE_RANK (best for the Nth highest).

### Common mistakes
- `ORDER BY salary DESC LIMIT 1 OFFSET 1` returns 80,000 here - wrong
- Using `RANK` instead of `DENSE_RANK` - rank 2 is skipped on ties
- Forgetting the NULL case when only one salary exists

## 10. Practice
**Task (from the video):** find the second highest salary in **each department**.

In [10]:
con.executescript("""
DROP TABLE IF EXISTS Staff;
CREATE TABLE Staff (name TEXT, dept TEXT, salary INTEGER);
INSERT INTO Staff VALUES
  ('Amit', 'IT', 50000), ('Priya', 'IT', 80000), ('Rahul', 'IT', 80000), ('Sneha', 'IT', 60000),
  ('Kiran', 'HR', 40000), ('Neha', 'HR', 45000), ('Vikram', 'Sales', 70000);
""")
sql("SELECT * FROM Staff ORDER BY dept, salary DESC")

,name,dept,salary
0,Neha,HR,45000
1,Kiran,HR,40000
2,Priya,IT,80000
3,Rahul,IT,80000
4,Sneha,IT,60000
5,Amit,IT,50000
6,Vikram,Sales,70000


In [11]:
# Solution: PARTITION BY restarts the ranking inside every department
sql("""
SELECT d.dept, MAX(r.salary) AS second_highest_salary
FROM (SELECT DISTINCT dept FROM Staff) d
LEFT JOIN (
    SELECT dept, salary,
           DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk
    FROM Staff
) r ON r.dept = d.dept AND r.rnk = 2
GROUP BY d.dept
ORDER BY d.dept
""")

,dept,second_highest_salary
0,HR,40000
1,IT,60000
2,Sales,<NA>


Sales has only one employee, so its answer is `NULL` (shown as `<NA>`) - the `LEFT JOIN` from the list of departments keeps it in the result.

**Next question:** SQL #2 - Derive the ICC Points Table.